In [1]:
!pip install -q google-generativeai pandas
!pip install -U google-generativeai

In [2]:
import google.generativeai as genai
import pandas as pd
import json
import re
from getpass import getpass

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
import google.generativeai as genai
from getpass import getpass

GOOGLE_API_KEY = getpass("Enter API Key: ")
genai.configure(api_key=GOOGLE_API_KEY)

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

Enter API Key: ··········
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-previe

In [13]:
GOOGLE_API_KEY = getpass("Enter your Gemini API Key: ")

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-3.5-flash")

print("Gemini Connected Successfully!")

Enter your Gemini API Key: ··········
Gemini Connected Successfully!


In [14]:
nutrition_db = {
    "rice": {"calories":130,"protein":2.7,"carbs":28,"fat":0.3,"diabetic":False},
    "brown rice":{"calories":111,"protein":2.6,"carbs":23,"fat":0.9,"diabetic":True},
    "chicken":{"calories":165,"protein":31,"carbs":0,"fat":3.6,"diabetic":True},
    "egg":{"calories":155,"protein":13,"carbs":1.1,"fat":11,"diabetic":True},
    "paneer":{"calories":265,"protein":18,"carbs":1.2,"fat":20,"diabetic":True},
    "tofu":{"calories":76,"protein":8,"carbs":2,"fat":4.8,"diabetic":True},
    "potato":{"calories":77,"protein":2,"carbs":17,"fat":0.1,"diabetic":False},
    "broccoli":{"calories":35,"protein":2.8,"carbs":7,"fat":0.4,"diabetic":True},
    "spinach":{"calories":23,"protein":2.9,"carbs":3.6,"fat":0.4,"diabetic":True},
    "tomato":{"calories":18,"protein":0.9,"carbs":3.9,"fat":0.2,"diabetic":True},
    "onion":{"calories":40,"protein":1.1,"carbs":9.3,"fat":0.1,"diabetic":True},
    "milk":{"calories":42,"protein":3.4,"carbs":5,"fat":1,"diabetic":True},
    "cheese":{"calories":402,"protein":25,"carbs":1.3,"fat":33,"diabetic":True},
    "carrot":{"calories":41,"protein":0.9,"carbs":10,"fat":0.2,"diabetic":True},
    "beans":{"calories":347,"protein":21,"carbs":63,"fat":1.2,"diabetic":True},
    "lentils":{"calories":116,"protein":9,"carbs":20,"fat":0.4,"diabetic":True},
    "banana":{"calories":89,"protein":1.1,"carbs":23,"fat":0.3,"diabetic":False},
    "apple":{"calories":52,"protein":0.3,"carbs":14,"fat":0.2,"diabetic":True},
    "oats":{"calories":389,"protein":17,"carbs":66,"fat":7,"diabetic":True},
    "bread":{"calories":265,"protein":9,"carbs":49,"fat":3.2,"diabetic":False}
}

In [6]:
def calculate_nutrition(ingredients):

    total = {
        "calories":0,
        "protein":0,
        "carbs":0,
        "fat":0
    }

    diabetic_safe = True

    for item in ingredients:

        item = item.lower().strip()

        if item in nutrition_db:

            data = nutrition_db[item]

            total["calories"] += data["calories"]
            total["protein"] += data["protein"]
            total["carbs"] += data["carbs"]
            total["fat"] += data["fat"]

            if not data["diabetic"]:
                diabetic_safe = False

    return total, diabetic_safe

In [7]:
def generate_recipe(ingredients,diet):

    prompt = f"""

You are an expert chef and nutritionist.

Generate a healthy recipe.

Diet Type:
{diet}

Ingredients:
{ingredients}

Return ONLY in this format.

Recipe Name:

Ingredients:

Instructions:

Estimated Nutrition:
Calories:
Protein:
Carbs:
Fat:

"""

    response = model.generate_content(prompt)

    return response.text

In [8]:
def validate_recipe(ingredients,
                    calories_target,
                    protein_target,
                    carb_target,
                    fat_target,
                    diabetic):

    nutrition, diabetic_safe = calculate_nutrition(ingredients)

    report=[]

    if abs(nutrition["calories"]-calories_target)>150:
        report.append("Calories not within target.")

    if abs(nutrition["protein"]-protein_target)>15:
        report.append("Protein target not satisfied.")

    if abs(nutrition["carbs"]-carb_target)>20:
        report.append("Carbohydrate target not satisfied.")

    if abs(nutrition["fat"]-fat_target)>10:
        report.append("Fat target not satisfied.")

    if diabetic and not diabetic_safe:
        report.append("Recipe is NOT diabetic-friendly.")

    if len(report)==0:
        report.append("Recipe passed all nutritional constraints.")

    return nutrition,report

In [1]:
ingredients = input("Enter ingredients separated by commas:\n")

ingredients = [x.strip() for x in ingredients.split(",")]

diet = input("Diet (Normal/Vegetarian/Vegan/Diabetic): ")

calories = int(input("Target Calories: "))

protein = int(input("Target Protein(g): "))

carbs = int(input("Target Carbs(g): "))

fat = int(input("Target Fat(g): "))

KeyboardInterrupt: Interrupted by user

In [15]:
recipe = generate_recipe(ingredients,diet)

print("="*80)
print(recipe)
print("="*80)

Recipe Name:
Savory Onion Egg Fried Rice

Ingredients:
- 1 cup cooked brown rice (leftover and chilled works best)
- 2 large eggs, whisked
- 1/2 medium onion, finely chopped
- 1 teaspoon olive oil
- 1 tablespoon low-sodium soy sauce (optional)
- Freshly ground black pepper, to taste

Instructions:
1. Heat the olive oil in a non-stick skillet or wok over medium heat.
2. Add the chopped onion to the pan and sauté for 3 to 4 minutes until it becomes soft and translucent.
3. Push the onions to one side of the skillet. Pour the whisked eggs into the empty side and scramble them gently until they are just set but still moist.
4. Add the cooked brown rice to the skillet, breaking up any clumps. Stir-fry everything together, mixing the rice, egg, and onion thoroughly.
5. Drizzle with the low-sodium soy sauce (if using) and season with freshly ground black pepper. Stir-fry for another 2 minutes until the rice is heated through and slightly toasted. 
6. Remove from heat and serve warm.

Estimate

In [16]:
nutrition,report = validate_recipe(
    ingredients,
    calories,
    protein,
    carbs,
    fat,
    diet.lower()=="diabetic"
)

In [17]:
print("\nNutritional Analysis")

df = pd.DataFrame([nutrition])

display(df)

print("\nValidation Report")

for r in report:
    print("•",r)


Nutritional Analysis


,calories,protein,carbs,fat
0,325,16.8,38.4,11.4



Validation Report
• Calories not within target.
• Protein target not satisfied.
